In [155]:
%load_ext autoreload
%autoreload 2 
    
from __init__ import PRP; import sys
sys.path.append(PRP + 'Diffusion_model')
sys.path.append(PRP + 'Results/analysis_scripts/')


from pipelines.pipeline_tensor import DDPMPipeline_Tensor
from generate_images import get_constraints
from pipelines.constraints import *

import torch, os
from visualisation_utils import *
dl, s = 4, 7

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


[autoreload of pipelines.constraints failed: Traceback (most recent call last):
  File "/Users/emeunier/miniforge3/envs/diffusion/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 276, in check
    superreload(m, reload, self.old_objects)
  File "/Users/emeunier/miniforge3/envs/diffusion/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 475, in superreload
    module = reload(module)
             ^^^^^^^^^^^^^^
  File "/Users/emeunier/miniforge3/envs/diffusion/lib/python3.12/importlib/__init__.py", line 131, in reload
    _bootstrap._exec(spec, module)
  File "<frozen importlib._bootstrap>", line 866, in _exec
  File "<frozen importlib._bootstrap_external>", line 991, in exec_module
  File "<frozen importlib._bootstrap_external>", line 1129, in get_code
  File "<frozen importlib._bootstrap_external>", line 1059, in source_to_code
  File "<frozen importlib._bootstrap>", line 488, in _call_with_frames_removed
  File "/Users/emeunier/Desktop/Projets/D

In [156]:
config = TrainingConfig()
config.data_file='/Volumes/LoCe/oceandata//Dino-Fusion/dino_1_4_degree_coarse_240125.tar'

## Data

In [157]:
next(idt).shape

OSError: [Errno 5] Input/output error

In [ ]:
train_dataloader = get_dataloader(config.data_file,
                                  batch_size=3,
                                  fields=config.fields,
                                  normalisation=config.normalisation, shuffle=False)

idt = iter(train_dataloader)

Data = State('data', config)
Data.normalized =  next(idt)

## Generate

In [ ]:
model_path = f'{os.environ["OCEANDATA"]}/models/dino-fusion/tav0h83b/'
beta = 1.0
beta_type = 'constant'
inf_steps = 1000

In [ ]:
a = torch.rand(3)- 0.5

In [ ]:
a[a>0].sum()

In [ ]:
pipeline = DDPMPipeline_Tensor.from_pretrained(model_path).to('mps')

In [158]:
pipeline.constraints = [BorderZeroConstraint(),
                        NegativeGradientDensityConstraint(beta=beta, beta_type=beta_type)]

Generated = State('generated_borderzero_negativegradientdensity', config)
Generated.normalized = pipeline(batch_size=3,
                                num_inference_steps=inf_steps,
                                return_dict=False)[0]

Reading infos in /Volumes/LoCe/oceandata//Dino-Fusion/dino_1_4_degree_coarse_240125_summer.tar
Reading infos in /Volumes/LoCe/oceandata//Dino-Fusion/dino_1_4_degree_coarse_240125.tar


  0%|          | 0/1000 [00:00<?, ?it/s]

apply constraint b 1.0: 0.0
apply constraint b 1.0: 0.0
apply constraint b 1.0: 0.0
apply constraint b 1.0: 0.0
apply constraint b 1.0: 0.0
apply constraint b 1.0: 0.0
apply constraint b 1.0: 0.0
apply constraint b 1.0: 0.0
apply constraint b 1.0: 0.0
apply constraint b 1.0: 0.0
apply constraint b 1.0: 0.0
apply constraint b 1.0: 0.0
apply constraint b 1.0: 0.0
apply constraint b 1.0: 0.0
apply constraint b 1.0: 0.0
apply constraint b 1.0: 0.0
apply constraint b 1.0: 0.0
apply constraint b 1.0: 0.0
apply constraint b 1.0: 0.0
apply constraint b 1.0: 0.0
apply constraint b 1.0: 0.0
apply constraint b 1.0: 0.0
apply constraint b 1.0: 0.0
apply constraint b 1.0: 0.0
apply constraint b 1.0: 0.0
apply constraint b 1.0: 0.0
apply constraint b 1.0: 0.0
apply constraint b 1.0: 0.0
apply constraint b 1.0: 0.0
apply constraint b 1.0: 0.0
apply constraint b 1.0: 0.0089111328125
apply constraint b 1.0000008344650269: 0.0045166015625
apply constraint b 1.000001311302185: 0.0177001953125
apply const

KeyboardInterrupt: 

In [ ]:
print(pipeline.constraints)
fig, axs = plt.subplots(1,dl, figsize=(20,3))
fig.suptitle(f'Surface temperature histogram - Generation {pipeline}')
for i, depth in enumerate(range(dl)) :
    axs[i].set_xlabel(f'layer = {depth}')
    for idx in range(1) :
        histogram(Generated.normalized[idx, depth, s:-s, s:-s].cpu(), ax=axs[i])
fig.legend()

In [ ]:
fig, axs = plt.subplots(2,2, figsize=(15,7))

fig.suptitle(f'Generated profiles {Generated.name}')
axs[0, 0].set_title('Density profile')
axs[0, 0].plot(Data.density.nanmean(axis=(-1,-2))[0, :-1], c='red', marker='o')
axs[0, 0].plot(Generated.density.nanmean(axis=(-1,-2))[:, :-1].T)

axs[0, 1].set_title('Density gradients')
axs[0, 1].plot(Data.density_gradient.nanmean(axis=(-1,-2))[0, :-1], c='red', marker='o')
axs[0, 1].plot(Generated.density_gradient.nanmean(axis=(-1,-2))[:, :-1].T)

axs[1, 0].set_title('Temperature profile')
axs[1, 0].plot(Data.unnormalized['toce.npy'].nanmean(axis=(-1,-2))[0, :-1].T, c='red', marker='o')
axs[1, 0].plot(Generated.unnormalized['toce.npy'].nanmean(axis=(-1,-2))[:, :-1].T)

axs[1, 1].set_title('Salinity profile')
axs[1, 1].plot(Data.unnormalized['soce.npy'].nanmean(axis=(-1,-2))[0, :-1].T, c='red', marker='o')
axs[1, 1].plot(Generated.unnormalized['soce.npy'].nanmean(axis=(-1,-2))[:, :-1].T)